This notebook compares the performance in the training and test dose-response curves datasets and pSTAT-dRNA correlations with Cui et. al.'s IL-10 DEG using different model methodologies (Linear, polynomial, splines and mechanistic). All models used the same initial conditions as our IL-10 signaling model (RA_0, RB_0, STAT1_0, STAT3_0 and IL10_0) as predictors, trained and tested in the same data as our mechanistic IL-10 signaling model.

In [60]:
from sklearn.linear_model import LinearRegression
from scipy.optimize import curve_fit
from src.simulate_func import hill_func,set_simulation,simulate_parallel_ODE
from src.models.IL10_RAp_ODE import model_function
import pandas as pd
import numpy as np
import seaborn as sns
from scipy.stats import spearmanr,pearsonr
import matplotlib.pyplot as plt
#from matplotlib import use
#use('Agg')
import warnings
warnings.filterwarnings('ignore')
plt.rc('xtick', labelsize=14)
plt.rc('ytick', labelsize=14)
from sklearn.metrics import r2_score

In [2]:
# Import datasets
path_data = '/users/lserrano/qmarti/PhD_code'
df_IC_data = pd.read_csv(path_data+'/IL10/data/expression/whole_dataset_IL10.tsv.gz', sep="\t", compression="gzip")
df_bind = pd.read_csv('data/binding/IL10_data_param_ABC_SMC_IL10_RAp_ODE.csv')
df_sim_data = pd.read_csv(path_data+'/IL10/data/signaling/IL10_STAT_data.tsv.gz', sep="\t", compression="gzip")
df_EC50_AMP = pd.read_csv(path_data+'/IL10/data/signaling/IL10_EC50_AMP.tsv.gz', sep="\t", compression="gzip")

# Divide data into train and tests datasets as done in the IL-10 signaling model
variants = ["WT","Super-10","R5A11D"]
cells = ["T8.Mean","T4.Mean","MO.Mean","ACH-000146","ACH-000786"]
df_sim_data_train = df_sim_data.loc[(df_sim_data["Variant"].isin(variants))&(df_sim_data["Cell_type"].isin(cells))]
df_sim_data_train.reset_index(drop=True, inplace=True)
df_EC50_AMP_train = df_EC50_AMP.loc[(df_EC50_AMP["Variant"].isin(variants))&(df_EC50_AMP["Cell_type"].isin(cells))]
df_EC50_AMP_train.reset_index(drop=True, inplace=True)
df_sim_data_test = pd.concat([df_sim_data.loc[(df_sim_data["Variant"].isin(variants))&(~df_sim_data["Cell_type"].isin(cells))], pd.read_csv('data/signaling/IL10_STAT_data_CD8_Gorby.tsv.gz', sep="\t", compression="gzip")], ignore_index=True)
df_sim_data_test.reset_index(drop=True, inplace=True)
df_sim_data_test = df_sim_data_test.loc[df_sim_data_test["Cell_type"]!="ACH-002317"]
df_EC50_AMP_test = pd.concat([df_EC50_AMP.loc[(df_EC50_AMP["Variant"].isin(variants))&(~df_EC50_AMP["Cell_type"].isin(cells))], pd.read_csv('data/signaling/IL10_EC50_AMP_CD8_Gorby.tsv.gz', sep="\t", compression="gzip")], ignore_index=True)
df_EC50_AMP_test.reset_index(drop=True, inplace=True)
df_EC50_AMP_test = df_EC50_AMP_test.loc[df_EC50_AMP_test["Cell_type"]!="ACH-002317"]

In [3]:
# Save all data of models in a dataframe
df_models = pd.DataFrame(columns=["Model","EC50e_train","AMPe_train","EC50e_test","AMPe_test","SpearmanR_up","SpearmanR_down"])

# Linear model only with receptor,STAT expression and IL10 concentration on multiple variants

In [4]:
# Organize data to train linear model
df_sim_train = df_IC_data.pivot(index='Cell', columns='Gene', values='Log10 Prot. count').loc[df_sim_data_train["Cell_type"]]
df_sim_train.columns = ["RA0","RB0","STAT10","STAT30"]
df_sim_train["IL0"] = np.log10(df_sim_data_train["IL"].values)
df_sim_train["STAT_type"] = (df_sim_data_train["STAT_type"] == "pSTAT1").astype(int).values

In [5]:
# Train linear model
lin_model_full = LinearRegression().fit(df_sim_train.values, df_sim_data_train["pSTAT"].values)

In [6]:
# Get error in EC50 and amplitude in training data
num_sim = 20
df_sim_train_full = set_simulation(df_bind, df_sim_data_train, df_IC_data, num_sim)
df_sim_train_full = df_sim_train_full[df_sim_train.columns]
df_sim_train_full["IL0"] = np.log10(df_sim_train_full["IL0"])
df_sim_train_full["STAT_type"] = (df_sim_train_full["STAT_type"] == "pSTAT1").astype(int).values
for plot_num in df_sim_data_train["Plot"].drop_duplicates():
    for variant in df_sim_data_train.loc[df_sim_data_train["Plot"]==plot_num,"Variant"].drop_duplicates():
        pSTAT5 = lin_model_full.predict(df_sim_train_full.loc[df_sim_data_train.loc[(df_sim_data_train["Plot"]==plot_num) & (df_sim_data_train["Variant"]==variant)].index].values)
        IL2 = 10**df_sim_train_full.loc[df_sim_data_train.loc[(df_sim_data_train["Plot"]==plot_num) & (df_sim_data_train["Variant"]==variant)].index,"IL0"].values
        fit, cov = curve_fit(hill_func, IL2, pSTAT5, bounds = ([min(pSTAT5),min(IL2)/10], [max(pSTAT5)+10, max(IL2)*10]))
        df_EC50_AMP_train.loc[(df_EC50_AMP_train["Plot"]==plot_num)&(df_EC50_AMP_train["Variant"]==variant),"EC50_m"] = np.log10(fit[1])
        df_EC50_AMP_train.loc[(df_EC50_AMP_train["Plot"]==plot_num)&(df_EC50_AMP_train["Variant"]==variant),"Amp_m"] = fit[0]
# Save errors
eEC50_train = (df_EC50_AMP_train["EC50"]-df_EC50_AMP_train["EC50_m"]).abs().mean()
eAMP_train = (df_EC50_AMP_train["Amp"]-df_EC50_AMP_train["Amp_m"]).abs().mean()
print("MAE_EC50: "+str(round(eEC50_train,2)))
print("MAE_Amplitude: "+str(round(eAMP_train,2)))

MAE_EC50: 1.0
MAE_Amplitude: 46.96


In [7]:
# Organize data to test linear model
df_sim_test = df_IC_data.pivot(index='Cell', columns='Gene', values='Log10 Prot. count').loc[df_sim_data_test["Cell_type"]]
df_sim_test.columns = ["RA0","RB0","STAT10","STAT30"]
df_sim_test["IL0"] = np.log10(df_sim_data_test["IL"].values)
df_sim_test["STAT_type"] = (df_sim_data_test["STAT_type"] == "pSTAT1").astype(int).values

In [8]:
# Get error in EC50 and amplitude in test data
num_sim = 20
df_sim_test_full = set_simulation(df_bind, df_sim_data_test, df_IC_data, num_sim)
df_sim_test_full = df_sim_test_full[df_sim_test.columns]
df_sim_test_full["IL0"] = np.log10(df_sim_test_full["IL0"])
df_sim_test_full["STAT_type"] = (df_sim_test_full["STAT_type"] == "pSTAT1").astype(int).values
for plot_num in df_sim_data_test["Plot"].drop_duplicates():
    for variant in df_sim_data_test.loc[df_sim_data_test["Plot"]==plot_num,"Variant"].drop_duplicates():
        pSTAT5 = lin_model_full.predict(df_sim_test_full.loc[df_sim_data_test.loc[(df_sim_data_test["Plot"]==plot_num) & (df_sim_data_test["Variant"]==variant)].index].values)
        IL2 = 10**df_sim_test_full.loc[df_sim_data_test.loc[(df_sim_data_test["Plot"]==plot_num) & (df_sim_data_test["Variant"]==variant)].index,"IL0"].values
        fit, cov = curve_fit(hill_func, IL2, pSTAT5, bounds = ([min(pSTAT5),min(IL2)/10], [max(pSTAT5)+10, max(IL2)*10]))
        df_EC50_AMP_test.loc[(df_EC50_AMP_test["Plot"]==plot_num)&(df_EC50_AMP_test["Variant"]==variant),"EC50_m"] = np.log10(fit[1])
        df_EC50_AMP_test.loc[(df_EC50_AMP_test["Plot"]==plot_num)&(df_EC50_AMP_test["Variant"]==variant),"Amp_m"] = fit[0]
# Save errors
eEC50_test = (df_EC50_AMP_test["EC50"]-df_EC50_AMP_test["EC50_m"]).abs().mean()
eAMP_test = (df_EC50_AMP_test["Amp"]-df_EC50_AMP_test["Amp_m"]).abs().mean()
print("MAE_EC50: "+str(round(eEC50_test,2)))
print("MAE_Amplitude: "+str(round(eAMP_test,2)))

MAE_EC50: 0.93
MAE_Amplitude: 29.03


In [9]:
# Get simulation data
df_sim3 = pd.read_csv("results/immune_dict/simulations_cytk_dict_RAp_ODE.csv")
df_sim3 = df_sim3.loc[df_sim3["IL0"]==2.437003460544915e-06]
cells = df_sim3["Cell_type"]
df_sim1 = pd.read_csv("results/immune_dict/simulations_cytk_dict_RAp_ODE_S1.csv")
df_sim1 = df_sim1.loc[df_sim1["IL0"]==2.437003460544915e-06]
df_sim3 = df_sim3[["RA0","RB0","STAT10","STAT30",'IL0',"STAT_type"]]
df_sim3["STAT_type"] = (df_sim3["STAT_type"] == "pSTAT1").astype(int).values
df_sim3["Result"] = lin_model_full.predict(df_sim3.values)
df_sim1 = df_sim1[["RA0","RB0","STAT10","STAT30",'IL0',"STAT_type"]]
df_sim1["STAT_type"] = (df_sim1["STAT_type"] == "pSTAT1").astype(int).values
df_sim1["Result"] = lin_model_full.predict(df_sim1.values)
df_sim3["Result"] = df_sim3["Result"]+df_sim1["Result"]
df_sim3.index = cells
# Get the mean RNA change per cell on upregulated and downregualted genes
df_diff_up = pd.read_csv("results/immune_dict/mean_dRNA_up.csv",index_col=0)
df_diff_down = pd.read_csv("results/immune_dict/mean_dRNA_down.csv",index_col=0)

print("Spearman's correlation coefficient for upregulated genes: "+str(round(spearmanr(df_sim3.loc[df_diff_up.index,"Result"],df_diff_up["0"])[0],2)))
rs_up = spearmanr(df_sim3.loc[df_diff_up.index,"Result"],df_diff_up["0"])[0]
print("Spearman's correlation coefficient for downregulated genes: "+str(round(spearmanr(df_sim3.loc[df_diff_down.index,"Result"],df_diff_down["0"])[0],2)))
rs_down = spearmanr(df_sim3.loc[df_diff_down.index,"Result"],df_diff_down["0"])[0]

Spearman's correlation coefficient for upregulated genes: -0.65
Spearman's correlation coefficient for downregulated genes: 0.7


In [10]:
# Save all metrics
df_models.loc[len(df_models.index)] = ["Linear_expr_full",eEC50_train,eAMP_train,eEC50_test,eAMP_test,rs_up,rs_down]

# Linear model only with receptor,STAT expression and IL10 concentration on WT data

In [11]:
# Train linear model
df_sim_train.index = df_sim_data_train.index
lin_model_WT = LinearRegression().fit(df_sim_train.loc[df_sim_data_train.loc[df_sim_data_train["Variant"]=="WT"].index].values, df_sim_data_train.loc[df_sim_data_train["Variant"]=="WT","pSTAT"].values)

In [12]:
# Get error in EC50 and amplitude in training data
num_sim = 20
df_sim_train_full = set_simulation(df_bind, df_sim_data_train, df_IC_data, num_sim)
df_sim_train_full = df_sim_train_full[df_sim_train.columns]
df_sim_train_full["IL0"] = np.log10(df_sim_train_full["IL0"])
df_sim_train_full["STAT_type"] = (df_sim_train_full["STAT_type"] == "pSTAT1").astype(int).values
for plot_num in df_sim_data_train["Plot"].drop_duplicates():
    variant = "WT"
    pSTAT5 = lin_model_WT.predict(df_sim_train_full.loc[df_sim_data_train.loc[(df_sim_data_train["Plot"]==plot_num) & (df_sim_data_train["Variant"]==variant)].index].values)
    IL2 = 10**df_sim_train_full.loc[df_sim_data_train.loc[(df_sim_data_train["Plot"]==plot_num) & (df_sim_data_train["Variant"]==variant)].index,"IL0"].values
    fit, cov = curve_fit(hill_func, IL2, pSTAT5, bounds = ([min(pSTAT5),min(IL2)/10], [max(pSTAT5)+10, max(IL2)*10]))
    df_EC50_AMP_train.loc[(df_EC50_AMP_train["Plot"]==plot_num)&(df_EC50_AMP_train["Variant"]==variant),"EC50_m"] = np.log10(fit[1])
    df_EC50_AMP_train.loc[(df_EC50_AMP_train["Plot"]==plot_num)&(df_EC50_AMP_train["Variant"]==variant),"Amp_m"] = fit[0]
# Save errors
eEC50_train = (df_EC50_AMP_train["EC50"]-df_EC50_AMP_train["EC50_m"]).abs().mean()
eAMP_train = (df_EC50_AMP_train["Amp"]-df_EC50_AMP_train["Amp_m"]).abs().mean()
print("MAE_EC50: "+str(round(eEC50_train,2)))
print("MAE_Amplitude: "+str(round(eAMP_train,2)))

MAE_EC50: 1.01
MAE_Amplitude: 38.16


In [13]:
# Get error in EC50 and amplitude in test data
df_sim_test.index = df_sim_data_test.index
num_sim = 20
df_sim_test_full = set_simulation(df_bind, df_sim_data_test, df_IC_data, num_sim)
df_sim_test_full = df_sim_test_full[df_sim_test.columns]
df_sim_test_full["IL0"] = np.log10(df_sim_test_full["IL0"])
df_sim_test_full["STAT_type"] = (df_sim_test_full["STAT_type"] == "pSTAT1").astype(int).values
for plot_num in df_sim_data_test["Plot"].drop_duplicates():
    variant = "WT"
    pSTAT5 = lin_model_full.predict(df_sim_test_full.loc[df_sim_data_test.loc[(df_sim_data_test["Plot"]==plot_num) & (df_sim_data_test["Variant"]==variant)].index].values)
    IL2 = 10**df_sim_test_full.loc[df_sim_data_test.loc[(df_sim_data_test["Plot"]==plot_num) & (df_sim_data_test["Variant"]==variant)].index,"IL0"].values
    fit, cov = curve_fit(hill_func, IL2, pSTAT5, bounds = ([min(pSTAT5),min(IL2)/10], [max(pSTAT5)+10, max(IL2)*10]))
    df_EC50_AMP_test.loc[(df_EC50_AMP_test["Plot"]==plot_num)&(df_EC50_AMP_test["Variant"]==variant),"EC50_m"] = np.log10(fit[1])
    df_EC50_AMP_test.loc[(df_EC50_AMP_test["Plot"]==plot_num)&(df_EC50_AMP_test["Variant"]==variant),"Amp_m"] = fit[0]
# Save errors
eEC50_test = (df_EC50_AMP_test["EC50"]-df_EC50_AMP_test["EC50_m"]).abs().mean()
eAMP_test = (df_EC50_AMP_test["Amp"]-df_EC50_AMP_test["Amp_m"]).abs().mean()
print("MAE_EC50: "+str(round(eEC50_test,2)))
print("MAE_Amplitude: "+str(round(eAMP_test,2)))

MAE_EC50: 0.93
MAE_Amplitude: 29.03


In [14]:
# Get simulation data
df_sim3 = pd.read_csv("results/immune_dict/simulations_cytk_dict_RAp_ODE.csv")
df_sim3 = df_sim3.loc[df_sim3["IL0"]==2.437003460544915e-06]
cells = df_sim3["Cell_type"]
df_sim1 = pd.read_csv("results/immune_dict/simulations_cytk_dict_RAp_ODE_S1.csv")
df_sim1 = df_sim1.loc[df_sim1["IL0"]==2.437003460544915e-06]
df_sim3 = df_sim3[["RA0","RB0","STAT10","STAT30",'IL0',"STAT_type"]]
df_sim3["STAT_type"] = (df_sim3["STAT_type"] == "pSTAT1").astype(int).values
df_sim3["Result"] = lin_model_WT.predict(df_sim3.values)
df_sim1 = df_sim1[["RA0","RB0","STAT10","STAT30",'IL0',"STAT_type"]]
df_sim1["STAT_type"] = (df_sim1["STAT_type"] == "pSTAT1").astype(int).values
df_sim1["Result"] = lin_model_WT.predict(df_sim1.values)
df_sim3["Result"] = df_sim3["Result"]+df_sim1["Result"]
df_sim3.index = cells
# Get the mean RNA change per cell on upregulated and downregualted genes
df_diff_up = pd.read_csv("results/immune_dict/mean_dRNA_up.csv",index_col=0)
df_diff_down = pd.read_csv("results/immune_dict/mean_dRNA_down.csv",index_col=0)

print("Spearman's correlation coefficient for upregulated genes: "+str(round(spearmanr(df_sim3.loc[df_diff_up.index,"Result"],df_diff_up["0"])[0],2)))
rs_up = spearmanr(df_sim3.loc[df_diff_up.index,"Result"],df_diff_up["0"])[0]
print("Spearman's correlation coefficient for downregulated genes: "+str(round(spearmanr(df_sim3.loc[df_diff_down.index,"Result"],df_diff_down["0"])[0],2)))
rs_down = spearmanr(df_sim3.loc[df_diff_down.index,"Result"],df_diff_down["0"])[0]

Spearman's correlation coefficient for upregulated genes: 0.18
Spearman's correlation coefficient for downregulated genes: -0.35


In [15]:
# Save all metrics
df_models.loc[len(df_models.index)] = ["Linear_expr_WT",eEC50_train,eAMP_train,eEC50_test,eAMP_test,rs_up,rs_down]

# Linear model only with receptor,STAT expression, IL2 concentration and Kd on multiple variants

In [16]:
# As with IL-2 signaling model, get the number of receptors and STAT proteins per cell type (training data)
df_sim_data_train_kd = pd.concat([df_bind.set_index("Variant").loc[df_sim_data_train["Variant"]].reset_index(drop=True)[df_bind.columns.to_list()[1:]],df_IC_data.pivot(index='Cell', columns='Gene', values='Log10 Prot. count').loc[df_sim_data_train["Cell_type"]].reset_index(drop=True)],axis=1)
df_sim_data_train_kd.columns = ['vol_EC', 'surf_cell', 'vol_cell', 'k_ILM_f', 'k_ILM_b', 'k_IL_RA_f',
       'k_IL_RA_b', 'k_IL_RB_f', 'k_IL_RB_b', 'k_IL_RA_RB_f', 'k_IL_RA_RB_b',
       'k_STAT1_f', 'k_STAT1_b', 'k_STAT3_f', 'k_STAT3_b', 'k_M_f', 'k_M_b',
       'k_PHOS', 'k_DEPHOS', 'k_DEPHOS_R', 'M0', "RA0","RB0","STAT10","STAT30"]
df_sim_data_train_kd[df_sim_data_train_kd.columns[:-5]] = np.log10(df_sim_data_train_kd[df_sim_data_train_kd.columns[:-5]])
df_sim_data_train_kd["IL0"] = np.log10(df_sim_data_train["IL"].values)
df_sim_data_train_kd["STAT_type"] = (df_sim_data_train["STAT_type"] == "pSTAT1").astype(int).values
df_sim_data_train_kd.replace([-np.inf], -1e-25, inplace=True)
df_sim_data_train_kd.replace([np.inf], 1e-25, inplace=True)

In [17]:
# Train linear model
lin_model_full_kd = LinearRegression().fit(df_sim_data_train_kd.values, df_sim_data_train["pSTAT"].values)

In [18]:
# Get error in EC50 and amplitude in training data
num_sim = 20
df_sim_train_kd = set_simulation(df_bind, df_sim_data_train, df_IC_data, num_sim)
df_sim_train_kd = df_sim_train_kd[df_sim_data_train_kd.columns]
df_sim_train_kd[df_sim_data_train_kd.columns[:-7]] = np.log10(df_sim_train_kd[df_sim_data_train_kd.columns[:-7]])
df_sim_data_train_kd["IL0"] = np.log10(df_sim_data_train["IL"].values)
df_sim_train_kd["STAT_type"] = (df_sim_train_kd["STAT_type"] == "pSTAT1").astype(int).values
df_sim_train_kd.replace([-np.inf], -1e-25, inplace=True)
df_sim_train_kd.replace([np.inf], 1e-25, inplace=True)
for plot_num in df_sim_data_train["Plot"].drop_duplicates():
    for variant in df_sim_data_train.loc[df_sim_data_train["Plot"]==plot_num,"Variant"].drop_duplicates():
        pSTAT5 = lin_model_full_kd.predict(df_sim_train_kd.loc[df_sim_data_train.loc[(df_sim_data_train["Plot"]==plot_num) & (df_sim_data_train["Variant"]==variant)].index].values)
        IL2 = 10**df_sim_train_kd.loc[df_sim_data_train.loc[(df_sim_data_train["Plot"]==plot_num) & (df_sim_data_train["Variant"]==variant)].index,"IL0"].values
        fit, cov = curve_fit(hill_func, IL2, pSTAT5, bounds = ([min(pSTAT5),min(IL2)/10], [max(pSTAT5)+10, max(IL2)*10]))
        df_EC50_AMP_train.loc[(df_EC50_AMP_train["Plot"]==plot_num)&(df_EC50_AMP_train["Variant"]==variant),"EC50_m"] = np.log10(fit[1])
        df_EC50_AMP_train.loc[(df_EC50_AMP_train["Plot"]==plot_num)&(df_EC50_AMP_train["Variant"]==variant),"Amp_m"] = fit[0]
# Save errors
eEC50_train = (df_EC50_AMP_train["EC50"]-df_EC50_AMP_train["EC50_m"]).abs().mean()
eAMP_train = (df_EC50_AMP_train["Amp"]-df_EC50_AMP_train["Amp_m"]).abs().mean()
print("MAE_EC50: "+str(round(eEC50_train,2)))
print("MAE_Amplitude: "+str(round(eAMP_train,2)))

MAE_EC50: 8.3
MAE_Amplitude: 250.75


In [19]:
# As with IL-2 signaling model, get the number of receptors and STAT proteins per cell type (testing data)
df_sim_data_test_kd = pd.concat([df_bind.set_index("Variant").loc[df_sim_data_test["Variant"]].reset_index(drop=True)[df_bind.columns.to_list()[1:]],df_IC_data.pivot(index='Cell', columns='Gene', values='Log10 Prot. count').loc[df_sim_data_test["Cell_type"]].reset_index(drop=True)],axis=1)
df_sim_data_test_kd.columns = ['vol_EC', 'surf_cell', 'vol_cell', 'k_ILM_f', 'k_ILM_b', 'k_IL_RA_f',
       'k_IL_RA_b', 'k_IL_RB_f', 'k_IL_RB_b', 'k_IL_RA_RB_f', 'k_IL_RA_RB_b',
       'k_STAT1_f', 'k_STAT1_b', 'k_STAT3_f', 'k_STAT3_b', 'k_M_f', 'k_M_b',
       'k_PHOS', 'k_DEPHOS', 'k_DEPHOS_R', 'M0', "RA0","RB0","STAT10","STAT30"]
df_sim_data_test_kd[df_sim_data_test_kd.columns[:-5]] = np.log10(df_sim_data_test_kd[df_sim_data_test_kd.columns[:-5]])
df_sim_data_test_kd["IL0"] = np.log10(df_sim_data_test["IL"].values)
df_sim_data_test_kd["STAT_type"] = (df_sim_data_test["STAT_type"] == "pSTAT1").astype(int).values
df_sim_data_test_kd.replace([-np.inf], -1e-25, inplace=True)
df_sim_data_test_kd.replace([np.inf], 1e-25, inplace=True)

In [20]:
# Get error in EC50 and amplitude in testing data
num_sim = 20
df_sim_test_kd = set_simulation(df_bind, df_sim_data_test, df_IC_data, num_sim)
df_sim_test_kd = df_sim_test_kd[df_sim_data_test_kd.columns]
df_sim_test_kd[df_sim_data_test_kd.columns[:-7]] = np.log10(df_sim_test_kd[df_sim_data_test_kd.columns[:-7]])
df_sim_data_test_kd["IL0"] = np.log10(df_sim_data_test["IL"].values)
df_sim_test_kd["STAT_type"] = (df_sim_test_kd["STAT_type"] == "pSTAT1").astype(int).values
df_sim_test_kd.replace([-np.inf], -1e-25, inplace=True)
df_sim_test_kd.replace([np.inf], 1e-25, inplace=True)
for plot_num in df_sim_data_test["Plot"].drop_duplicates():
    for variant in df_sim_data_test.loc[df_sim_data_test["Plot"]==plot_num,"Variant"].drop_duplicates():
        pSTAT5 = lin_model_full_kd.predict(df_sim_test_kd.loc[df_sim_data_test.loc[(df_sim_data_test["Plot"]==plot_num) & (df_sim_data_test["Variant"]==variant)].index].values)
        IL2 = 10**df_sim_test_kd.loc[df_sim_data_test.loc[(df_sim_data_test["Plot"]==plot_num) & (df_sim_data_test["Variant"]==variant)].index,"IL0"].values
        fit, cov = curve_fit(hill_func, IL2, pSTAT5, bounds = ([min(pSTAT5),min(IL2)/10], [max(pSTAT5)+10, max(IL2)*10]))
        df_EC50_AMP_test.loc[(df_EC50_AMP_test["Plot"]==plot_num)&(df_EC50_AMP_test["Variant"]==variant),"EC50_m"] = np.log10(fit[1])
        df_EC50_AMP_test.loc[(df_EC50_AMP_test["Plot"]==plot_num)&(df_EC50_AMP_test["Variant"]==variant),"Amp_m"] = fit[0]
# Save errors
eEC50_test = (df_EC50_AMP_test["EC50"]-df_EC50_AMP_test["EC50_m"]).abs().mean()
eAMP_test = (df_EC50_AMP_test["Amp"]-df_EC50_AMP_test["Amp_m"]).abs().mean()
print("MAE_EC50: "+str(round(eEC50_test,2)))
print("MAE_Amplitude: "+str(round(eAMP_test,2)))

MAE_EC50: 8.24
MAE_Amplitude: 244.84


In [21]:
# Get simulation data
df_sim3 = pd.read_csv("results/immune_dict/simulations_cytk_dict_RAp_ODE.csv")
df_sim3 = df_sim3.loc[df_sim3["IL0"]==2.437003460544915e-06]
cells = df_sim3["Cell_type"]
df_sim1 = pd.read_csv("results/immune_dict/simulations_cytk_dict_RAp_ODE_S1.csv")
df_sim1 = df_sim1.loc[df_sim1["IL0"]==2.437003460544915e-06]
df_sim3 = df_sim3[df_sim_test_kd.columns]
df_sim3["STAT_type"] = (df_sim3["STAT_type"] == "pSTAT1").astype(int).values
df_sim3["Result"] = lin_model_full_kd.predict(df_sim3.values)
df_sim1 = df_sim1[df_sim_test_kd.columns]
df_sim1["STAT_type"] = (df_sim1["STAT_type"] == "pSTAT1").astype(int).values
df_sim1["Result"] = lin_model_full_kd.predict(df_sim1.values)
df_sim3["Result"] = df_sim3["Result"]+df_sim1["Result"]
df_sim3.index = cells
# Get the mean RNA change per cell on upregulated and downregualted genes
df_diff_up = pd.read_csv("results/immune_dict/mean_dRNA_up.csv",index_col=0)
df_diff_down = pd.read_csv("results/immune_dict/mean_dRNA_down.csv",index_col=0)

print("Spearman's correlation coefficient for upregulated genes: "+str(round(spearmanr(df_sim3.loc[df_diff_up.index,"Result"],df_diff_up["0"])[0],2)))
rs_up = spearmanr(df_sim3.loc[df_diff_up.index,"Result"],df_diff_up["0"])[0]
print("Spearman's correlation coefficient for downregulated genes: "+str(round(spearmanr(df_sim3.loc[df_diff_down.index,"Result"],df_diff_down["0"])[0],2)))
rs_down = spearmanr(df_sim3.loc[df_diff_down.index,"Result"],df_diff_down["0"])[0]

Spearman's correlation coefficient for upregulated genes: -0.65
Spearman's correlation coefficient for downregulated genes: 0.7


In [22]:
# Save all metrics
df_models.loc[len(df_models.index)] = ["Linear_expr_Kd_full",eEC50_train,eAMP_train,eEC50_test,eAMP_test,rs_up,rs_down]

# Polynomic model only with receptor,STAT5 expression, IL2 concentration and Kd on multiple variants

In [23]:
from sklearn.pipeline import make_pipeline
from sklearn.preprocessing import SplineTransformer,PolynomialFeatures
from sklearn.linear_model import Ridge

In [24]:
for alpha in [0.001, 0.01, 0.1, 1.0, 10, 100, 1000, 10000, 100000, 1000000]:
    model_poly_full_kd = make_pipeline(PolynomialFeatures(degree=2), Ridge(alpha=alpha))
    model_poly_full_kd = model_poly_full_kd.fit(df_sim_data_train_kd.values, df_sim_data_train["pSTAT"].values)
    print(alpha)
    # Get error in EC50 and amplitude in training data
    for plot_num in df_sim_data_train["Plot"].drop_duplicates():
        for variant in df_sim_data_train.loc[df_sim_data_train["Plot"]==plot_num,"Variant"].drop_duplicates():
            pSTAT5 = model_poly_full_kd.predict(df_sim_train_kd.loc[df_sim_data_train.loc[(df_sim_data_train["Plot"]==plot_num) & (df_sim_data_train["Variant"]==variant)].index].values)
            IL2 = 10**df_sim_train_kd.loc[df_sim_data_train.loc[(df_sim_data_train["Plot"]==plot_num) & (df_sim_data_train["Variant"]==variant)].index,"IL0"].values
            fit, cov = curve_fit(hill_func, IL2, pSTAT5, bounds = ([min(pSTAT5),min(IL2)/10], [max(pSTAT5)+10, max(IL2)*10]))
            df_EC50_AMP_train.loc[(df_EC50_AMP_train["Plot"]==plot_num)&(df_EC50_AMP_train["Variant"]==variant),"EC50_m"] = np.log10(fit[1])
            df_EC50_AMP_train.loc[(df_EC50_AMP_train["Plot"]==plot_num)&(df_EC50_AMP_train["Variant"]==variant),"Amp_m"] = fit[0]
    # Save errors
    eEC50_train = (df_EC50_AMP_train["EC50"]-df_EC50_AMP_train["EC50_m"]).abs().mean()
    eAMP_train = (df_EC50_AMP_train["Amp"]-df_EC50_AMP_train["Amp_m"]).abs().mean()
    print("MAE_EC50: "+str(round(eEC50_train,2)))
    print("MAE_Amplitude: "+str(round(eAMP_train,2)))

0.001
MAE_EC50: 8.34
MAE_Amplitude: 290.73
0.01
MAE_EC50: 8.32
MAE_Amplitude: 272.02
0.1
MAE_EC50: 8.32
MAE_Amplitude: 272.8
1.0
MAE_EC50: 8.32
MAE_Amplitude: 277.19
10
MAE_EC50: 8.31
MAE_Amplitude: 270.37
100
MAE_EC50: 8.4
MAE_Amplitude: 147.31
1000
MAE_EC50: 8.3
MAE_Amplitude: 143.42
10000
MAE_EC50: 8.31
MAE_Amplitude: 237.97
100000
MAE_EC50: 8.3
MAE_Amplitude: 249.44
1000000
MAE_EC50: 8.3
MAE_Amplitude: 250.61


In [25]:
model_poly_full_kd = make_pipeline(PolynomialFeatures(degree=2), Ridge(alpha=1000)) # Take alpha as 1000 as it has the lowest error in amplitude (error in EC50 is bad in all)
model_poly_full_kd = model_poly_full_kd.fit(df_sim_data_train_kd.values, df_sim_data_train["pSTAT"].values)

In [26]:
# Get error in EC50 and amplitude in training data
for plot_num in df_sim_data_train["Plot"].drop_duplicates():
    for variant in df_sim_data_train.loc[df_sim_data_train["Plot"]==plot_num,"Variant"].drop_duplicates():
        pSTAT5 = model_poly_full_kd.predict(df_sim_train_kd.loc[df_sim_data_train.loc[(df_sim_data_train["Plot"]==plot_num) & (df_sim_data_train["Variant"]==variant)].index].values)
        IL2 = 10**df_sim_train_kd.loc[df_sim_data_train.loc[(df_sim_data_train["Plot"]==plot_num) & (df_sim_data_train["Variant"]==variant)].index,"IL0"].values
        fit, cov = curve_fit(hill_func, IL2, pSTAT5, bounds = ([min(pSTAT5),min(IL2)/10], [max(pSTAT5)+10, max(IL2)*10]))
        df_EC50_AMP_train.loc[(df_EC50_AMP_train["Plot"]==plot_num)&(df_EC50_AMP_train["Variant"]==variant),"EC50_m"] = np.log10(fit[1])
        df_EC50_AMP_train.loc[(df_EC50_AMP_train["Plot"]==plot_num)&(df_EC50_AMP_train["Variant"]==variant),"Amp_m"] = fit[0]
# Save errors
eEC50_train = (df_EC50_AMP_train["EC50"]-df_EC50_AMP_train["EC50_m"]).abs().mean()
eAMP_train = (df_EC50_AMP_train["Amp"]-df_EC50_AMP_train["Amp_m"]).abs().mean()
print("MAE_EC50: "+str(round(eEC50_train,2)))
print("MAE_Amplitude: "+str(round(eAMP_train,2)))

MAE_EC50: 8.3
MAE_Amplitude: 143.42


In [27]:
for plot_num in df_sim_data_test["Plot"].drop_duplicates():
    for variant in df_sim_data_test.loc[df_sim_data_test["Plot"]==plot_num,"Variant"].drop_duplicates():
        pSTAT5 = model_poly_full_kd.predict(df_sim_test_kd.loc[df_sim_data_test.loc[(df_sim_data_test["Plot"]==plot_num) & (df_sim_data_test["Variant"]==variant)].index].values)
        IL2 = 10**df_sim_test_kd.loc[df_sim_data_test.loc[(df_sim_data_test["Plot"]==plot_num) & (df_sim_data_test["Variant"]==variant)].index,"IL0"].values
        fit, cov = curve_fit(hill_func, IL2, pSTAT5, bounds = ([min(pSTAT5),min(IL2)/10], [max(pSTAT5)+10, max(IL2)*10]))
        df_EC50_AMP_test.loc[(df_EC50_AMP_test["Plot"]==plot_num)&(df_EC50_AMP_test["Variant"]==variant),"EC50_m"] = np.log10(fit[1])
        df_EC50_AMP_test.loc[(df_EC50_AMP_test["Plot"]==plot_num)&(df_EC50_AMP_test["Variant"]==variant),"Amp_m"] = fit[0]
# Save errors
eEC50_test = (df_EC50_AMP_test["EC50"]-df_EC50_AMP_test["EC50_m"]).abs().mean()
eAMP_test = (df_EC50_AMP_test["Amp"]-df_EC50_AMP_test["Amp_m"]).abs().mean()
print("MAE_EC50: "+str(round(eEC50_test,2)))
print("MAE_Amplitude: "+str(round(eAMP_test,2)))

MAE_EC50: 8.25
MAE_Amplitude: 139.37


In [28]:
# Get simulation data
df_sim3 = pd.read_csv("results/immune_dict/simulations_cytk_dict_RAp_ODE.csv")
df_sim3 = df_sim3.loc[df_sim3["IL0"]==2.437003460544915e-06]
cells = df_sim3["Cell_type"]
df_sim1 = pd.read_csv("results/immune_dict/simulations_cytk_dict_RAp_ODE_S1.csv")
df_sim1 = df_sim1.loc[df_sim1["IL0"]==2.437003460544915e-06]
df_sim3 = df_sim3[df_sim_test_kd.columns]
df_sim3["STAT_type"] = (df_sim3["STAT_type"] == "pSTAT1").astype(int).values
df_sim3["Result"] = model_poly_full_kd.predict(df_sim3.values)
df_sim1 = df_sim1[df_sim_test_kd.columns]
df_sim1["STAT_type"] = (df_sim1["STAT_type"] == "pSTAT1").astype(int).values
df_sim1["Result"] = model_poly_full_kd.predict(df_sim1.values)
df_sim3["Result"] = df_sim3["Result"]+df_sim1["Result"]
df_sim3.index = cells
# Get the mean RNA change per cell on upregulated and downregualted genes
df_diff_up = pd.read_csv("results/immune_dict/mean_dRNA_up.csv",index_col=0)
df_diff_down = pd.read_csv("results/immune_dict/mean_dRNA_down.csv",index_col=0)

print("Spearman's correlation coefficient for upregulated genes: "+str(round(spearmanr(df_sim3.loc[df_diff_up.index,"Result"],df_diff_up["0"])[0],2)))
rs_up = spearmanr(df_sim3.loc[df_diff_up.index,"Result"],df_diff_up["0"])[0]
print("Spearman's correlation coefficient for downregulated genes: "+str(round(spearmanr(df_sim3.loc[df_diff_down.index,"Result"],df_diff_down["0"])[0],2)))
rs_down = spearmanr(df_sim3.loc[df_diff_down.index,"Result"],df_diff_down["0"])[0]

Spearman's correlation coefficient for upregulated genes: 0.61
Spearman's correlation coefficient for downregulated genes: -0.41


In [29]:
# Save all metrics
df_models.loc[len(df_models.index)] = ["Poly_expr_Kd_full",eEC50_train,eAMP_train,eEC50_test,eAMP_test,rs_up,rs_down]

# Spline regression with Kd

In [30]:
for alpha in [0.001, 0.01, 0.1, 1.0, 10, 100, 1000, 10000, 100000, 1000000]:
    model_spline_full_kd = make_pipeline(SplineTransformer(n_knots=4, degree=2), Ridge(alpha=alpha))
    model_spline_full_kd = model_spline_full_kd.fit(df_sim_data_train_kd.values, df_sim_data_train["pSTAT"].values)
    # Get error in EC50 and amplitude in training data
    for plot_num in df_sim_data_train["Plot"].drop_duplicates():
        for variant in df_sim_data_train.loc[df_sim_data_train["Plot"]==plot_num,"Variant"].drop_duplicates():
            pSTAT5 = model_spline_full_kd.predict(df_sim_train_kd.loc[df_sim_data_train.loc[(df_sim_data_train["Plot"]==plot_num) & (df_sim_data_train["Variant"]==variant)].index].values)
            IL2 = 10**df_sim_train_kd.loc[df_sim_data_train.loc[(df_sim_data_train["Plot"]==plot_num) & (df_sim_data_train["Variant"]==variant)].index,"IL0"].values
            fit, cov = curve_fit(hill_func, IL2, pSTAT5, bounds = ([min(pSTAT5),min(IL2)/10], [max(pSTAT5)+10, max(IL2)*10]))
            df_EC50_AMP_train.loc[(df_EC50_AMP_train["Plot"]==plot_num)&(df_EC50_AMP_train["Variant"]==variant),"EC50_m"] = np.log10(fit[1])
            df_EC50_AMP_train.loc[(df_EC50_AMP_train["Plot"]==plot_num)&(df_EC50_AMP_train["Variant"]==variant),"Amp_m"] = fit[0]
    # Save errors
    eEC50_train = (df_EC50_AMP_train["EC50"]-df_EC50_AMP_train["EC50_m"]).abs().mean()
    eAMP_train = (df_EC50_AMP_train["Amp"]-df_EC50_AMP_train["Amp_m"]).abs().mean()
    print(alpha)
    print("MAE_EC50: "+str(round(eEC50_train,2)))
    print("MAE_Amplitude: "+str(round(eAMP_train,2)))

0.001
MAE_EC50: 8.4
MAE_Amplitude: 22.57
0.01
MAE_EC50: 8.4
MAE_Amplitude: 22.56
0.1
MAE_EC50: 8.4
MAE_Amplitude: 22.48
1.0
MAE_EC50: 8.4
MAE_Amplitude: 21.79
10
MAE_EC50: 8.36
MAE_Amplitude: 21.86
100
MAE_EC50: 8.49
MAE_Amplitude: 29.26
1000
MAE_EC50: 8.44
MAE_Amplitude: 36.0
10000
MAE_EC50: 8.44
MAE_Amplitude: 36.58
100000
MAE_EC50: 8.44
MAE_Amplitude: 36.64
1000000
MAE_EC50: 8.47
MAE_Amplitude: 36.15


In [31]:
model_spline_full_kd = make_pipeline(SplineTransformer(n_knots=4, degree=2), Ridge(alpha=1))
model_spline_full_kd = model_spline_full_kd.fit(df_sim_data_train_kd.values, df_sim_data_train["pSTAT"].values)

In [32]:
# Get error in EC50 and amplitude in training data
for plot_num in df_sim_data_train["Plot"].drop_duplicates():
    for variant in df_sim_data_train.loc[df_sim_data_train["Plot"]==plot_num,"Variant"].drop_duplicates():
        pSTAT5 = model_spline_full_kd.predict(df_sim_train_kd.loc[df_sim_data_train.loc[(df_sim_data_train["Plot"]==plot_num) & (df_sim_data_train["Variant"]==variant)].index].values)
        IL2 = 10**df_sim_train_kd.loc[df_sim_data_train.loc[(df_sim_data_train["Plot"]==plot_num) & (df_sim_data_train["Variant"]==variant)].index,"IL0"].values
        fit, cov = curve_fit(hill_func, IL2, pSTAT5, bounds = ([min(pSTAT5),min(IL2)/10], [max(pSTAT5)+10, max(IL2)*10]))
        df_EC50_AMP_train.loc[(df_EC50_AMP_train["Plot"]==plot_num)&(df_EC50_AMP_train["Variant"]==variant),"EC50_m"] = np.log10(fit[1])
        df_EC50_AMP_train.loc[(df_EC50_AMP_train["Plot"]==plot_num)&(df_EC50_AMP_train["Variant"]==variant),"Amp_m"] = fit[0]
# Save errors
eEC50_train = (df_EC50_AMP_train["EC50"]-df_EC50_AMP_train["EC50_m"]).abs().mean()
eAMP_train = (df_EC50_AMP_train["Amp"]-df_EC50_AMP_train["Amp_m"]).abs().mean()
print("MAE_EC50: "+str(round(eEC50_train,2)))
print("MAE_Amplitude: "+str(round(eAMP_train,2)))

MAE_EC50: 8.4
MAE_Amplitude: 21.79


In [33]:
for plot_num in df_sim_data_test["Plot"].drop_duplicates():
    for variant in df_sim_data_test.loc[df_sim_data_test["Plot"]==plot_num,"Variant"].drop_duplicates():
        pSTAT5 = model_spline_full_kd.predict(df_sim_test_kd.loc[df_sim_data_test.loc[(df_sim_data_test["Plot"]==plot_num) & (df_sim_data_test["Variant"]==variant)].index].values)
        IL2 = 10**df_sim_test_kd.loc[df_sim_data_test.loc[(df_sim_data_test["Plot"]==plot_num) & (df_sim_data_test["Variant"]==variant)].index,"IL0"].values
        fit, cov = curve_fit(hill_func, IL2, pSTAT5, bounds = ([min(pSTAT5),min(IL2)/10], [max(pSTAT5)+10, max(IL2)*10]))
        df_EC50_AMP_test.loc[(df_EC50_AMP_test["Plot"]==plot_num)&(df_EC50_AMP_test["Variant"]==variant),"EC50_m"] = np.log10(fit[1])
        df_EC50_AMP_test.loc[(df_EC50_AMP_test["Plot"]==plot_num)&(df_EC50_AMP_test["Variant"]==variant),"Amp_m"] = fit[0]
# Save errors
eEC50_test = (df_EC50_AMP_test["EC50"]-df_EC50_AMP_test["EC50_m"]).abs().mean()
eAMP_test = (df_EC50_AMP_test["Amp"]-df_EC50_AMP_test["Amp_m"]).abs().mean()
print("MAE_EC50: "+str(round(eEC50_test,2)))
print("MAE_Amplitude: "+str(round(eAMP_test,2)))

MAE_EC50: 8.32
MAE_Amplitude: 12.17


In [34]:
# Get simulation data
df_sim3 = pd.read_csv("results/immune_dict/simulations_cytk_dict_RAp_ODE.csv")
df_sim3 = df_sim3.loc[df_sim3["IL0"]==2.437003460544915e-06]
cells = df_sim3["Cell_type"]
df_sim1 = pd.read_csv("results/immune_dict/simulations_cytk_dict_RAp_ODE_S1.csv")
df_sim1 = df_sim1.loc[df_sim1["IL0"]==2.437003460544915e-06]
df_sim3 = df_sim3[df_sim_test_kd.columns]
df_sim3["STAT_type"] = (df_sim3["STAT_type"] == "pSTAT1").astype(int).values
df_sim3["Result"] = model_spline_full_kd.predict(df_sim3.values)
df_sim1 = df_sim1[df_sim_test_kd.columns]
df_sim1["STAT_type"] = (df_sim1["STAT_type"] == "pSTAT1").astype(int).values
df_sim1["Result"] = model_spline_full_kd.predict(df_sim1.values)
df_sim3["Result"] = df_sim3["Result"]+df_sim1["Result"]
df_sim3.index = cells
# Get the mean RNA change per cell on upregulated and downregualted genes
df_diff_up = pd.read_csv("results/immune_dict/mean_dRNA_up.csv",index_col=0)
df_diff_down = pd.read_csv("results/immune_dict/mean_dRNA_down.csv",index_col=0)

print("Spearman's correlation coefficient for upregulated genes: "+str(round(spearmanr(df_sim3.loc[df_diff_up.index,"Result"],df_diff_up["0"])[0],2)))
rs_up = spearmanr(df_sim3.loc[df_diff_up.index,"Result"],df_diff_up["0"])[0]
print("Spearman's correlation coefficient for downregulated genes: "+str(round(spearmanr(df_sim3.loc[df_diff_down.index,"Result"],df_diff_down["0"])[0],2)))
rs_down = spearmanr(df_sim3.loc[df_diff_down.index,"Result"],df_diff_down["0"])[0]

Spearman's correlation coefficient for upregulated genes: -0.43
Spearman's correlation coefficient for downregulated genes: 0.32


In [35]:
# Save all metrics
df_models.loc[len(df_models.index)] = ["Spline_expr_Kd_full",eEC50_train,eAMP_train,eEC50_test,eAMP_test,rs_up,rs_down]

In [36]:
df_models

,Model,EC50e_train,AMPe_train,EC50e_test,AMPe_test,SpearmanR_up,SpearmanR_down
0,Linear_expr_full,1.004043,46.958778,0.932006,29.029268,-0.654545,0.700000
1,Linear_expr_WT,1.014572,38.164924,0.932006,29.029268,0.181818,-0.345455
2,Linear_expr_Kd_full,8.303712,250.746266,8.244373,244.841465,-0.654545,0.700000
3,Poly_expr_Kd_full,8.297014,143.419162,8.254360,139.366717,0.609091,-0.409091
4,Spline_expr_Kd_full,8.403948,21.788034,8.324286,12.167230,-0.427273,0.318182


# Mechanistic IL-10 signaling model

In [40]:
# Get simualtions for the specific [IL-10] for training and test dataset
df_sim_mech = pd.read_csv("results/fit_param/simulations_fit_IL10_RAp_ODE.csv")
df_sim_mech["Plot__Variant"] = df_sim_mech["Plot"].astype("str")+"__"+df_sim_mech["Variant"].astype("str")
for plot_num in df_sim_mech["Plot"].drop_duplicates():
    max_WT = df_sim_mech.loc[(df_sim_mech["Plot"]==plot_num) & (df_sim_mech["Variant"]=="WT"),"Result"].max()
    df_sim_mech.loc[df_sim_mech["Plot"]==plot_num,"Result"] = df_sim_mech.loc[df_sim_mech["Plot"]==plot_num,"Result"]/max_WT*100
df_sim_mech_train = df_sim_mech.loc[df_sim_mech["Plot"].isin(df_sim_data_train["Plot"].drop_duplicates().values)]
df_sim_mech_test = df_sim_mech.loc[df_sim_mech["Plot"].isin(df_sim_data_test["Plot"].drop_duplicates().values)]

In [61]:
# For each celltype/variant simulated
plot_num_old = 0
for key in df_sim_mech_train["Plot__Variant"]:
    plot_num, variant = key.split("__")
    plot_num = int(plot_num)
    df_var = df_sim_mech_train.loc[(df_sim_mech_train["Plot"] == plot_num)&(df_sim_mech_train["Variant"] == variant)]
    # Data is normalized to the first variant that appears in a plot (Determined from df_sim to be WT)
    if variant == "WT":
        max_WT = df_var["Result"].max()
        fit, cov = curve_fit(hill_func, df_var["IL0"], df_var["Result"].values*100/(max_WT), bounds = ([0,df_var["IL0"].min()/10], [100, df_var["IL0"].max()*10]))
        df_EC50_AMP_train.loc[(df_EC50_AMP_train["Plot"]==plot_num)&(df_EC50_AMP_train["Variant"]==variant),"Amp_m"] = fit[0]
        df_EC50_AMP_train.loc[(df_EC50_AMP_train["Plot"]==plot_num)&(df_EC50_AMP_train["Variant"]==variant),"EC50_m"] = np.log10(fit[1])
    else:
        fit, cov = curve_fit(hill_func, df_var["IL0"], df_var["Result"].values*100/(max_WT), bounds = ([0,df_var["IL0"].min()/10], [df_var["Result"].max()*100/(max_WT)+5, df_var["IL0"].max()*10]))
        df_EC50_AMP_train.loc[(df_EC50_AMP_train["Plot"]==plot_num)&(df_EC50_AMP_train["Variant"]==variant),"Amp_m"] = fit[0]
        df_EC50_AMP_train.loc[(df_EC50_AMP_train["Plot"]==plot_num)&(df_EC50_AMP_train["Variant"]==variant),"EC50_m"] = np.log10(fit[1])
    plot_num_old = plot_num

eEC50_train = (df_EC50_AMP_train["EC50"]-df_EC50_AMP_train["EC50_m"]).abs().mean()
eAMP_train = (df_EC50_AMP_train["Amp"]-df_EC50_AMP_train["Amp_m"]).abs().mean()
print("MAE_EC50: "+str(round(eEC50_train,2)))
print("MAE_Amplitude: "+str(round(eAMP_train,2)))

MAE_EC50: 0.27
MAE_Amplitude: 6.2


In [62]:
# For each celltype/variant simulated
plot_num_old = 0
for key in df_sim_mech_test["Plot__Variant"]:
    plot_num, variant = key.split("__")
    plot_num = int(plot_num)
    df_var = df_sim_mech_test.loc[(df_sim_mech_test["Plot"] == plot_num)&(df_sim_mech_test["Variant"] == variant)]
    # Data is normalized to the first variant that appears in a plot (Determined from df_sim to be WT)
    if variant == "WT":
        max_WT = df_var["Result"].max()
        fit, cov = curve_fit(hill_func, df_var["IL0"], df_var["Result"].values*100/(max_WT), bounds = ([0,df_var["IL0"].min()/10], [100, df_var["IL0"].max()*10]))
        df_EC50_AMP_test.loc[(df_EC50_AMP_test["Plot"]==plot_num)&(df_EC50_AMP_test["Variant"]==variant),"Amp_m"] = fit[0]
        df_EC50_AMP_test.loc[(df_EC50_AMP_test["Plot"]==plot_num)&(df_EC50_AMP_test["Variant"]==variant),"EC50_m"] = np.log10(fit[1])
    else:
        fit, cov = curve_fit(hill_func, df_var["IL0"], df_var["Result"].values*100/(max_WT), bounds = ([0,df_var["IL0"].min()/10], [df_var["Result"].max()*100/(max_WT)+5, df_var["IL0"].max()*10]))
        df_EC50_AMP_test.loc[(df_EC50_AMP_test["Plot"]==plot_num)&(df_EC50_AMP_test["Variant"]==variant),"Amp_m"] = fit[0]
        df_EC50_AMP_test.loc[(df_EC50_AMP_test["Plot"]==plot_num)&(df_EC50_AMP_test["Variant"]==variant),"EC50_m"] = np.log10(fit[1])
    plot_num_old = plot_num

eEC50_test = (df_EC50_AMP_test["EC50"]-df_EC50_AMP_test["EC50_m"]).abs().mean()
eAMP_test = (df_EC50_AMP_test["Amp"]-df_EC50_AMP_test["Amp_m"]).abs().mean()
print("MAE_EC50: "+str(round(eEC50_test,2)))
print("MAE_Amplitude: "+str(round(eAMP_test,2)))

MAE_EC50: 0.15
MAE_Amplitude: 5.68


In [55]:
# Get simulation data
df_sim3 = pd.read_csv("results/immune_dict/simulations_cytk_dict_RAp_ODE.csv")
df_sim3 = df_sim3.loc[df_sim3["IL0"]==2.437003460544915e-06]
cells = df_sim3["Cell_type"]
df_sim1 = pd.read_csv("results/immune_dict/simulations_cytk_dict_RAp_ODE_S1.csv")
df_sim1 = df_sim1.loc[df_sim1["IL0"]==2.437003460544915e-06]
df_sim3["Result"] = df_sim3["Result"]+df_sim1["Result"]
df_sim3.index = cells

# Get the mean RNA change per cell on upregulated and downregualted genes
df_diff_up = pd.read_csv("results/immune_dict/mean_dRNA_up.csv",index_col=0)
df_diff_down = pd.read_csv("results/immune_dict/mean_dRNA_down.csv",index_col=0)

print("Spearman's correlation coefficient for upregulated genes: "+str(round(spearmanr(df_sim3.loc[df_diff_up.index,"Result"],df_diff_up["0"])[0],2)))
rs_up = spearmanr(df_sim3.loc[df_diff_up.index,"Result"],df_diff_up["0"])[0]
print("Spearman's correlation coefficient for downregulated genes: "+str(round(spearmanr(df_sim3.loc[df_diff_down.index,"Result"],df_diff_down["0"])[0],2)))
rs_down = spearmanr(df_sim3.loc[df_diff_down.index,"Result"],df_diff_down["0"])[0]

Spearman's correlation coefficient for upregulated genes: 0.89
Spearman's correlation coefficient for downregulated genes: -0.77


In [56]:
# Save all metrics
df_models.loc[len(df_models.index)] = ["Mech_model",eEC50_train,eAMP_train,eEC50_test,eAMP_test,rs_up,rs_down]

In [58]:
df_models

,Model,EC50e_train,AMPe_train,EC50e_test,AMPe_test,SpearmanR_up,SpearmanR_down
0,Linear_expr_full,1.004043,46.958778,0.932006,29.029268,-0.654545,0.700000
1,Linear_expr_WT,1.014572,38.164924,0.932006,29.029268,0.181818,-0.345455
2,Linear_expr_Kd_full,8.303712,250.746266,8.244373,244.841465,-0.654545,0.700000
3,Poly_expr_Kd_full,8.297014,143.419162,8.254360,139.366717,0.609091,-0.409091
4,Spline_expr_Kd_full,8.403948,21.788034,8.324286,12.167230,-0.427273,0.318182
5,Mech_model,0.268030,6.202117,0.148873,5.680497,0.890909,-0.772727
